In [41]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

from langchain_ollama import ChatOllama

class State(TypedDict):
    documents: list[str]
    context : str
    question: str
    answer: str

llm = ChatOllama(
    model= "llama3.2:1b",
    temperature=0,
)


In [ ]:
from langchain_core.tools import tool

@tool
def maths(a: float, b: float, operation: str) -> float:
    """
    Perform a mathematical operation on two numbers.

    Args:
        a: First number.
        b: Second number.
        operation: Operation to perform. Supported values:
                   add, subtract, multiply, divide.
    """
    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

    else:
        raise ValueError("Unsupported operation")

llm_with_tools = llm.bind_tools(tools=[maths])

respone = llm_with_tools.invoke("what is 5 plus 6")

tool_func_bind = {
    "maths" : maths
}

result = respone.tool_calls[0]

print(result)

func = tool_func_bind[result["name"]]
arsg = result["args"]
print(*arsg)
func.invoke(arsg)


{'name': 'maths', 'args': {'a': 5, 'b': 6, 'operation': 'add'}, 'id': '6a501303-dca9-433f-a5ea-43f75ba62afd', 'type': 'tool_call'}
a b operation


11.0

In [43]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embedding = OllamaEmbeddings(model="nomic-embed-text:latest")

loader = PyPDFLoader("/home/aakash/Projects/AI/interview-practice/src/docs/04-Policy-Manual-Apr-2016-excl-SAICA.pdf")

document = loader.load()


textSplitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = textSplitter.split_documents(document)


vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    collection_name="Policy_document",
    persist_directory="./chroma_db"

)

vector_store.similarity_search(query="How many leaved in calender year?",k=2)

retriever = vector_store.as_retriever(kwargs=3)



fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KLQRNP+Times-Roman', '/FontDescriptor': IndirectObject(10, 0, 127785193395792), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 95, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 500], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/ONXHWD+Helvetica-Bold', '/FontDescriptor': IndirectObject(12, 0, 127785193395792), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 146, '/Widths': [278, 0, 0, 0, 0, 0, 722, 0, 333, 333, 0, 0, 278, 333, 278, 278, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 333, 0, 0, 0, 0, 0, 0, 722, 722, 722, 722

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import llms
from langgraph.prebuilt import ToolNode, tools_condition

def retrieve(state: State):
    question = state['question']
    docs = retriever.invoke(question)
    return {
        "documents" : docs
    }


def generate(state: State):
    question = state["question"]
    documents = state["documents"]

    context = "\n\n".join(
        doc.page_content for doc in documents
    )


    prompt = f"""
    You are a helpfull AI assistant answering question based on the company policy documents
    Use only the following context to answer.
    If answer not provided in the context.
    say , I dont know.

    Rules:
        Do not assume.
        Do not hallucinate.

    Context:
        {context}
    Question:
        {question}
    """

    response = llm.invoke(prompt)
    return {"answer" : response.content}


In [45]:
graph = StateGraph(State)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

workflow = graph.compile()